# Payments Table Data Cleaning

Problems to Investigate
- Orders with 0 Payment Installments
- Payments with type set as 'not_defined'

In [2]:
import pandas as pd

payments_raw = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
payments = payments_raw.copy()

print("Original Shape:", payments_raw.shape)
print("Working Shape:", payments.shape)

Original Shape: (103886, 5)
Working Shape: (103886, 5)


In [3]:
# Orders with 0 Payment Installments

payments[payments["payment_installments"] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [4]:
# Payments with type 'not_defined'

payments[payments["payment_type"] == 'not_defined']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [5]:
# Are all payments with 0 value not defined

zero_value_payments = payments[
    payments["payment_value"] == 0
]

print("Total zero-value payment records:", len(zero_value_payments))

print("\nPayment types among zero-value records:")
print(zero_value_payments["payment_type"].value_counts())

print("\nInstallment distribution:")
print(zero_value_payments["payment_installments"].value_counts())

Total zero-value payment records: 9

Payment types among zero-value records:
payment_type
voucher        6
not_defined    3
Name: count, dtype: int64

Installment distribution:
payment_installments
1    9
Name: count, dtype: int64


In [6]:
# Do the 0 value voucher records have other payment records

zero_voucher_orders = payments.loc[
    (payments["payment_value"] == 0) &
    (payments["payment_type"] == "voucher"),
    "order_id"
].unique()

voucher_order_records = payments[
    payments["order_id"].isin(zero_voucher_orders)
].sort_values(["order_id", "payment_sequential"])

voucher_order_records

,order_id,payment_sequential,payment_type,payment_installments,payment_value
33781,45ed6e85398a87c253db47c2d9f48216,1,voucher,1,21.13
11755,45ed6e85398a87c253db47c2d9f48216,2,voucher,1,50.01
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
40546,6ccb433e00daae1283ccc956189c82ae,1,credit_card,5,84.67
93478,6ccb433e00daae1283ccc956189c82ae,2,voucher,1,14.65
92318,6ccb433e00daae1283ccc956189c82ae,3,voucher,1,22.72
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
20963,8bcbe01d44d147f901cd3192671144db,1,credit_card,1,36.21
63762,8bcbe01d44d147f901cd3192671144db,2,voucher,1,31.70
5163,8bcbe01d44d147f901cd3192671144db,3,voucher,1,6.25


In [7]:
# Do any of the 0 installment or undefined payment type orders overlap

zero_installment_orders = payments.loc[
    payments["payment_installments"] == 0, "order_id"
].unique()

undefined_payment_orders = payments.loc[
    payments["payment_type"] == "not_defined", "order_id"
].unique()

print(
    "Orders appearing in both groups:",
    len(set(zero_installment_orders) & set(undefined_payment_orders))
)

Orders appearing in both groups: 0


In [8]:
# Do affected orders have multiple payment records 

affected_orders = set(zero_installment_orders) | set(undefined_payment_orders)

affected_payment_records = payments[
    payments["order_id"].isin(affected_orders)
].sort_values(["order_id", "payment_sequential"])

affected_payment_records

,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00


In [9]:
print("Payment Types:")
print(payments["payment_type"].value_counts(dropna=False))

print("\nInstallment Distribution:")
print(payments["payment_installments"].value_counts(dropna=False).sort_index())

Payment Types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Installment Distribution:
payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64


In [10]:
print("Negative Payment Values:",
      (payments["payment_value"] < 0).sum())

print("Negative Installments:",
      (payments["payment_installments"] < 0).sum())

print("Zero Payment Values:",
      (payments["payment_value"] == 0).sum())

Negative Payment Values: 0
Negative Installments: 0
Zero Payment Values: 9


In [11]:
pd.crosstab(
    payments["payment_type"],
    payments["payment_installments"]
)

payment_installments,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,20,21,22,23,24
payment_type,,,,,,,,,,,,,,,,,,,,,
boleto,0,19784,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
credit_card,2,25455,12413,10461,7098,5239,3920,1626,4268,644,...,15,74,5,8,27,17,3,1,1,18
debit_card,0,1529,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
not_defined,0,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
voucher,0,5775,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
# Are there any duplicate payment records

print(
    "Duplicate payment records:",
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

Duplicate payment records: 0


In [13]:
# Are all payment sequences valid

sequence_summary = payments.groupby("order_id")[
    "payment_sequential"
].agg(["min", "max", "count"])

sequence_summary.head()

,min,max,count
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,1,1,1
00018f77f2f0320c557190d7a144bdd3,1,1,1
000229ec398224ef6ca0657da4fc703e,1,1,1
00024acbcdf0a6daa1e931b038114c75,1,1,1
00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1


In [14]:
# Do all payment sequences begin at 1

sequence_summary[
    sequence_summary["min"] != 1
]

,min,max,count
order_id,,,
00ac05fe0fc047c54418098eb64e3aaa,2,2,1
056c68d093c100017aab1f00f260705c,2,2,1
0668d086b3ae41adde3aed3dacbc8fae,2,2,1
0a7d898c6305101e69e9f5a05cd0130d,2,2,1
0d0e88a418636d40ea780339701db49c,2,2,1
...,...,...,...
f0f94b7c7548150f33f5d9e7597d396f,2,2,1
f17a58e669a7f4cf7a7f227899f8abf7,2,2,1
f5691c2b1ca263490374d13d020bd950,2,2,1


In [15]:
# Payments with sequences not starting at 1

orders_missing_sequence_1 = sequence_summary[
    sequence_summary["min"] != 1
].index

missing_sequence_records = payments[
    payments["order_id"].isin(orders_missing_sequence_1)
].sort_values("order_id")

missing_sequence_records.head(20)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
27191,00ac05fe0fc047c54418098eb64e3aaa,2,debit_card,1,123.47
12785,056c68d093c100017aab1f00f260705c,2,debit_card,1,52.78
40318,0668d086b3ae41adde3aed3dacbc8fae,2,debit_card,1,252.74
84637,0a7d898c6305101e69e9f5a05cd0130d,2,credit_card,5,215.47
82233,0d0e88a418636d40ea780339701db49c,2,debit_card,1,121.09
62609,0fdb6f73fab7005c84d4a43d8a93682b,2,debit_card,1,34.13
88530,159da9914b51de617fe80162939965c5,2,debit_card,1,40.76
66314,166b01d0fce75a595b92ad9e84e0843b,2,credit_card,6,127.09
87490,1896e8b8fd196151559b02e31e98ce89,2,debit_card,1,108.95
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [16]:
# Payment types of orders with sequences not starting at 1

print("Payment types:")
print(missing_sequence_records["payment_type"].value_counts())

print("\nPayment value summary:")
print(missing_sequence_records["payment_value"].describe())

print("\nInstallment distribution:")
print(missing_sequence_records["payment_installments"].value_counts())

Payment types:
payment_type
debit_card     52
credit_card    29
boleto          1
Name: count, dtype: int64

Payment value summary:
count     82.000000
mean     163.949878
std      153.827137
min       20.000000
25%       63.582500
50%      122.280000
75%      203.527500
max      847.380000
Name: payment_value, dtype: float64

Installment distribution:
payment_installments
1     55
8      6
3      5
10     4
4      4
5      2
6      2
0      2
2      2
Name: count, dtype: int64


In [17]:
# Number of payments 
print("Number of affected orders:",
      missing_sequence_records["order_id"].nunique())

print("Number of affected payment records:",
      len(missing_sequence_records))

print("\nPayment records per affected order:")
print(
    missing_sequence_records["order_id"]
    .value_counts()
    .value_counts()
)


Number of affected orders: 80
Number of affected payment records: 82

Payment records per affected order:
count
1    78
2     2
Name: count, dtype: int64


In [18]:
multiple_affected_orders = (
    missing_sequence_records["order_id"]
    .value_counts()
)

multiple_affected_orders = multiple_affected_orders[
    multiple_affected_orders > 1
].index

missing_sequence_records[
    missing_sequence_records["order_id"].isin(
        multiple_affected_orders
    )
].sort_values(["order_id", "payment_sequential"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
36471,a079628ac8002126e75f86b0f87332e4,2,debit_card,1,50.00
10258,a079628ac8002126e75f86b0f87332e4,3,credit_card,3,102.82
60570,a4431cbd79dbddaae7988ce6091cbc3c,2,debit_card,1,26.20
90465,a4431cbd79dbddaae7988ce6091cbc3c,3,debit_card,1,20.00


In [19]:
duplicate_sequences = payments[
    payments.duplicated(
        subset=["order_id", "payment_sequential"],
        keep=False
    )
]

print("Duplicate order-sequence combinations:", len(duplicate_sequences))

Duplicate order-sequence combinations: 0


In [20]:
sequence_gaps = payments.groupby("order_id")[
    "payment_sequential"
].apply(
    lambda x: set(range(1, x.max() + 1)) - set(x)
)

sequence_gaps[sequence_gaps.apply(len) > 0]

order_id
00ac05fe0fc047c54418098eb64e3aaa    {1}
056c68d093c100017aab1f00f260705c    {1}
0668d086b3ae41adde3aed3dacbc8fae    {1}
0a7d898c6305101e69e9f5a05cd0130d    {1}
0d0e88a418636d40ea780339701db49c    {1}
                                   ... 
f0f94b7c7548150f33f5d9e7597d396f    {1}
f17a58e669a7f4cf7a7f227899f8abf7    {1}
f5691c2b1ca263490374d13d020bd950    {1}
fce3db7092e1f432d0ec790414b6a5e4    {1}
fe49c4a1416840f8ed6bd36daac85aae    {1}
Name: payment_sequential, Length: 80, dtype: object

In [21]:
# Do all payments reference an order from the Orders dataset

orders = pd.read_csv('../data/processed/olist_orders_cleaned.csv')
payment_order_ids = set(payments["order_id"])
valid_order_ids = set(orders["order_id"])

unmatched_payment_orders = payment_order_ids - valid_order_ids

print("Payment Orders Not Found in Orders Table:",
      len(unmatched_payment_orders))

Payment Orders Not Found in Orders Table: 0


In [22]:
# Add quality flags 
 
payments["zero_payment_installments"] = (
    payments["payment_installments"] == 0
)

payments["undefined_payment_type"] = (
    payments["payment_type"] == "not_defined"
)

payments["zero_payment_value"] = (
    payments["payment_value"] == 0
)

payment_quality_flags = [
    "zero_payment_installments",
    "undefined_payment_type"
]

payments["has_data_quality_issue"] = (
    payments[payment_quality_flags].any(axis=1)
)

In [23]:
# Number of payments with sequences not starting at 1

payments["sequence_does_not_start_at_1"] = (
    payments.groupby("order_id")["payment_sequential"]
    .transform("min") != 1
)

print(
    "Affected payment records:",
    payments["sequence_does_not_start_at_1"].sum()
)

print(
    "Affected orders:",
    payments.loc[
        payments["sequence_does_not_start_at_1"], "order_id"
    ].nunique()
)

Affected payment records: 82
Affected orders: 80


In [24]:
# Remove any whitespace from column names

payments.columns = (
    payments.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(payments.columns.tolist())

# Remove any whitespace from string values

string_columns = payments.select_dtypes(
    include=["object", "str"]
).columns

for column in string_columns:
    payments[column] = payments[column].str.strip()

['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'zero_payment_installments', 'undefined_payment_type', 'zero_payment_value', 'has_data_quality_issue', 'sequence_does_not_start_at_1']


In [25]:
# Data quality flags

# Zero payment installments
payments["zero_payment_installments"] = (
    payments["payment_installments"] == 0
)

# Undefined payment type
payments["undefined_payment_type"] = (
    payments["payment_type"] == "not_defined"
)

# Zero payment value
payments["zero_payment_value"] = (
    payments["payment_value"] == 0
)

# Payment sequence does not start at 1
payments["sequence_does_not_start_at_1"] = (
    payments.groupby("order_id")["payment_sequential"]
    .transform("min") != 1
)

payment_quality_flags = [
    "zero_payment_installments",
    "undefined_payment_type"
]

payments["has_data_quality_issue"] = (
    payments[payment_quality_flags].any(axis=1)
)

In [26]:
# Check quality flags

flag_columns = [
    "zero_payment_installments",
    "undefined_payment_type",
    "zero_payment_value",
    "sequence_does_not_start_at_1",
    "has_data_quality_issue"
]

flag_summary = pd.DataFrame({
    "Flag": flag_columns,
    "Count": [
        payments[column].sum()
        for column in flag_columns
    ],
    "Percentage": [
        payments[column].mean() * 100
        for column in flag_columns
    ]
})

flag_summary["Percentage"] = flag_summary["Percentage"].round(3)

flag_summary

,Flag,Count,Percentage
0,zero_payment_installments,2,0.002
1,undefined_payment_type,3,0.003
2,zero_payment_value,9,0.009
3,sequence_does_not_start_at_1,82,0.079
4,has_data_quality_issue,5,0.005


In [27]:
print("Final Dataset Shape:", payments.shape)

print("\nMissing Values:")
print(payments.isnull().sum())

print("\nDuplicate Rows:", payments.duplicated().sum())

print("\nDuplicate Order/Sequence Combinations:",
      payments.duplicated(
          subset=["order_id", "payment_sequential"]
      ).sum())

print("\nData Types:")
print(payments.dtypes)

Final Dataset Shape: (103886, 10)

Missing Values:
order_id                        0
payment_sequential              0
payment_type                    0
payment_installments            0
payment_value                   0
zero_payment_installments       0
undefined_payment_type          0
zero_payment_value              0
has_data_quality_issue          0
sequence_does_not_start_at_1    0
dtype: int64

Duplicate Rows: 0

Duplicate Order/Sequence Combinations: 0

Data Types:
order_id                            str
payment_sequential                int64
payment_type                        str
payment_installments              int64
payment_value                   float64
zero_payment_installments          bool
undefined_payment_type             bool
zero_payment_value                 bool
has_data_quality_issue             bool
sequence_does_not_start_at_1       bool
dtype: object


In [28]:
# Export cleaned dataset

payments.to_csv(
    "../data/processed/olist_order_payments_cleaned.csv",
    index=False
)